# 24 — Multi-Turn Conversational RAG

> **Tier 5 | Multi-turn & Memory**

## What is Multi-Turn Conversational RAG?

NB 23 stored conversation history as plain text and injected recent turns into the prompt.
**Multi-Turn Conversational RAG** goes further: conversation turns are **embedded and stored
in a second Qdrant collection**, making history *semantically searchable*.

At query time the system retrieves from **both stores** and merges the results:

| Source | What it provides |
|--------|------------------|
| Document KB | Factual passages from the PDF |
| Conversation History Store | Semantically relevant past Q&A turns |

This matters when a user asks a question that was partially answered 10 turns ago —
a sliding window misses it, but semantic search finds it.

### Key differences vs NB 23

| Dimension | NB 23 Memory-Augmented | NB 24 Multi-Turn Conversational |
|-----------|----------------------|--------------------------------|
| History storage | In-memory list | Qdrant collection (embedded) |
| History retrieval | Last N turns (window) | Semantic search over all turns |
| Pronoun resolution | Text injection | Semantic retrieval finds relevant turns |
| Scalability | Degrades with long sessions | Scales to unlimited history |

## PDF Framework: pdfplumber

This notebook uses **pdfplumber** with `page.extract_words()` — word-level bounding-box
extraction. Words are grouped into lines by proximity, then chunked. This produces
cleaner sentence boundaries than page-level `extract_text()`.

| Feature | pdfplumber `extract_words()` |
|---------|-----------------------------|
| Granularity | Word-level with x/y coordinates |
| Line grouping | By `top` coordinate proximity |
| Layout awareness | Yes — respects column breaks |
| Previous use in series | NB 18 (Corrective RAG) used `bbox_density` |

## PDF Framework Progression (Qdrant series)

| NB | Pattern | Framework | Key API |
|----|---------|-----------|----------|
| 18 | Corrective RAG | **pdfplumber** | `bbox_density` |
| 19 | Self RAG | **pymupdf** `get_text("dict")` | Font size/bold → headings |
| 20 | Iterative RAG | **pdfminer.six** | `LTTextBox.y0` zone tagging |
| 21 | Recursive RAG | **pdfminer.six** heuristic | Element-type hierarchy |
| 22 | Agentic RAG | **pymupdf** `get_text("blocks")` | Block-type filter, block_no |
| 23 | Memory-Augmented RAG | **pypdf** `PdfReader` | `page.extract_text()` |
| **24** | **Multi-Turn Conversational RAG** | **pdfplumber** `extract_words()` | Word-level line grouping |

## Flow Diagram


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfplumber extract_words"]
        PDF(["📄 climate.pdf"])
        PDF --> PL["pdfplumber\npage.extract_words()"]
        PL --> LN["Word→Line grouping\n(top coordinate)"]
        LN --> CH["Text chunks"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> DOCSTORE[("Qdrant\nconv_rag_24_docs")]
    end
    subgraph HISTSTORE ["🧠  CONVERSATION HISTORY STORE"]
        HQDRANT[("Qdrant\nconv_rag_24_history")]
    end
    subgraph RAG ["🤖  MULTI-TURN CONVERSATIONAL RAG"]
        Q(["❓ Query"])
        Q --> EMBQ["Embed query"]
        EMBQ --> DOCSTORE
        EMBQ --> HQDRANT
        DOCSTORE --> MERGE["Merge results\n(doc chunks + past turns)"]
        HQDRANT --> MERGE
        MERGE --> GEN["LLM Generation\nClaude Sonnet 4.6"]
        GEN --> ANS(["✅ Answer"])
        ANS --> EMBA["Embed turn"]
        EMBA --> HQDRANT
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style HISTSTORE fill:#fef3c7,stroke:#f59e0b,color:#78350f
    style RAG fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pdfplumber", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid
from typing import List, Dict, Tuple
from dotenv import load_dotenv

import boto3
import pdfplumber
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

DOC_COLLECTION  = "conv_rag_24_docs"
HIST_COLLECTION = "conv_rag_24_history"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
DOC_TOP_K       = 4   # chunks from document store
HIST_TOP_K      = 2   # turns from history store

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Doc collection  : {DOC_COLLECTION}")
print(f"Hist collection : {HIST_COLLECTION}")
print(f"PDF             : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Retrieval mix   : {DOC_TOP_K} doc chunks + {HIST_TOP_K} history turns")


## Step 3 — Qdrant Client (shared)

In [ ]:
def make_qdrant_client(qdrant_url='', qdrant_api_key='',
                       opensearch_url='', region='us-east-1'):
    if qdrant_url:
        try:
            kw = {'url': qdrant_url}
            if qdrant_api_key: kw['api_key'] = qdrant_api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {qdrant_url}')
            return client, 'qdrant_cloud'
        except Exception as e:
            print(f'Qdrant Cloud unavailable ({e}), falling back...')
    client = QdrantClient(':memory:')
    print('Using Qdrant in-memory')
    return client, 'qdrant_memory'


def ensure_collection(client: QdrantClient, name: str, dim: int,
                      force_recreate: bool = False):
    exists = name in [c.name for c in client.get_collections().collections]
    if exists and force_recreate:
        client.delete_collection(name); exists = False
    if not exists:
        client.create_collection(
            name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
        print(f'Created "{name}" (dim={dim})')
    else:
        print(f'"{name}" already exists')


qdrant, backend = make_qdrant_client(
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {backend}")

ensure_collection(qdrant, DOC_COLLECTION,  EMBEDDING_DIM, force_recreate=True)
ensure_collection(qdrant, HIST_COLLECTION, EMBEDDING_DIM, force_recreate=True)


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

def call_llm(system: str, user_content: str, max_tokens: int = 1024) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]

test_emb = embed_text("multi turn conversational rag pdfplumber test")
print(f"Embedding OK — dim={len(test_emb)}")
print("call_llm ready.")


## Step 5 — PDF Loading with pdfplumber (`extract_words()`)

`pdfplumber` returns words as dicts with `{text, x0, top, ...}`. We group words
into lines by the `top` coordinate (±2 pt tolerance), join lines into paragraphs,
then split into fixed-size chunks. This is cleaner than raw `extract_text()` for
multi-column PDFs because column breaks are handled by the x/y grouping.


In [ ]:
def words_to_lines(words: List[Dict], y_tol: float = 2.0) -> List[str]:
    if not words:
        return []
    lines, cur_line, cur_top = [], [], None
    for w in sorted(words, key=lambda w: (round(w['top'] / y_tol), w['x0'])):
        top = round(w['top'] / y_tol)
        if cur_top is None or top != cur_top:
            if cur_line:
                lines.append(' '.join(cur_line))
            cur_line = [w['text']]
            cur_top  = top
        else:
            cur_line.append(w['text'])
    if cur_line:
        lines.append(' '.join(cur_line))
    return lines


def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pdfplumber(path: str):
    chunks = []
    with pdfplumber.open(path) as pdf:
        n_pages = len(pdf.pages)
        for page_num, page in enumerate(pdf.pages):
            words = page.extract_words()
            if not words:
                continue
            lines = words_to_lines(words)
            text  = ' '.join(lines).strip()
            if not text:
                continue
            for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
                chunks.append({
                    'text'      : sub,
                    'page'      : page_num + 1,
                    'char_count': len(sub),
                })

    stats = {
        'n_pages' : n_pages,
        'n_chunks': len(chunks),
        'avg_chars': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
doc_chunks, stats = load_pdf_pdfplumber(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pdfplumber extraction : {elapsed:.0f}ms")
print(f"Pages                 : {stats['n_pages']}")
print(f"Chunks                : {stats['n_chunks']}")
print(f"Avg chunk length      : {stats['avg_chars']} chars")


## Step 6 — Index Document Store

In [ ]:
print(f"Embedding {len(doc_chunks)} document chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in doc_chunks], label='[docs]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embs[i],
        payload={
            'text'    : doc_chunks[i]['text'],
            'metadata': {
                'page'      : doc_chunks[i]['page'],
                'char_count': doc_chunks[i]['char_count'],
                'source'    : 'climate.pdf',
                'pdf_lib'   : 'pdfplumber',
                'type'      : 'document',
            }
        }
    )
    for i in range(len(doc_chunks))
]
qdrant.upsert(collection_name=DOC_COLLECTION, points=pts)
doc_count = qdrant.get_collection(DOC_COLLECTION).points_count
print(f"Indexed {doc_count} document vectors in {time.time()-t0:.1f}s")


## Step 7 — Dual-Store Retrieval & History Indexing

Each completed Q&A turn is embedded as `"Q: {question} A: {answer}"` and upserted
into the history collection. At query time, both collections are searched with the
same query vector, and results are merged — document chunks labeled `[doc]`,
history turns labeled `[hist]`.


In [ ]:
def search_collection(collection: str, qvec: List[float], top_k: int) -> List[Dict]:
    resp = qdrant.query_points(
        collection_name=collection, query=qvec,
        limit=top_k, with_payload=True)
    return [{'text'    : p.payload.get('text', ''),
             'metadata': p.payload.get('metadata', {}),
             'score'   : p.score} for p in resp.points]


def store_turn(session_id: str, question: str, answer: str) -> None:
    turn_text = f"Q: {question}\nA: {answer}"
    vec = embed_text(turn_text)
    qdrant.upsert(
        collection_name=HIST_COLLECTION,
        points=[PointStruct(
            id=str(uuid.uuid4()),
            vector=vec,
            payload={
                'text'    : turn_text,
                'metadata': {
                    'session_id': session_id,
                    'question'  : question,
                    'type'      : 'history',
                }
            }
        )]
    )


def dual_retrieve(qvec: List[float]) -> Tuple[List[Dict], List[Dict]]:
    doc_hits  = search_collection(DOC_COLLECTION,  qvec, DOC_TOP_K)
    hist_hits = search_collection(HIST_COLLECTION, qvec, HIST_TOP_K)
    return doc_hits, hist_hits


print("Dual-store retrieval helpers defined.")
print(f"  doc  top-k : {DOC_TOP_K}")
print(f"  hist top-k : {HIST_TOP_K}")


## Step 8 — Multi-Turn Conversational RAG Pipeline

Pipeline per turn:
1. Embed query
2. Retrieve `DOC_TOP_K` document chunks + `HIST_TOP_K` history turns in parallel
3. Build prompt: history turns (labeled `[HISTORY]`) + doc chunks (labeled `[DOC]`) + question
4. LLM generates answer
5. Embed `"Q: ... A: ..."` and store in history collection


In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about a climate document.

You receive two types of context:
- [HISTORY] blocks: semantically relevant past Q&A turns from this session
- [DOC] blocks: passages retrieved from the document

Rules:
- Prioritise [DOC] passages for factual claims.
- Use [HISTORY] to resolve references and avoid repeating information already given.
- If the answer cannot be found in either, say so explicitly.
- Be concise and precise.
"""


def multi_turn_rag(
    question: str,
    session_id: str,
    turn_number: int,
    verbose: bool = True,
) -> Dict:
    t0 = time.time()

    # 1. Embed query
    qvec = embed_text(question)

    # 2. Dual retrieval
    doc_hits, hist_hits = dual_retrieve(qvec)

    # 3. Build context block
    hist_block = '\n\n'.join(
        f"[HISTORY {i+1}] score={h['score']:.3f}\n{h['text']}"
        for i, h in enumerate(hist_hits)
    ) if hist_hits else "(No relevant history yet)"

    doc_block = '\n\n'.join(
        f"[DOC {i+1}] page={h['metadata'].get('page','?')} score={h['score']:.3f}\n{h['text']}"
        for i, h in enumerate(doc_hits)
    )

    user_msg = f"""Relevant conversation history:
{hist_block}

Document context:
{doc_block}

Question: {question}"""

    # 4. Generate answer
    answer  = call_llm(SYSTEM_PROMPT, user_msg)
    latency = (time.time() - t0) * 1000

    # 5. Store turn in history collection
    store_turn(session_id, question, answer)

    if verbose:
        hist_count = qdrant.get_collection(HIST_COLLECTION).points_count or 0
        print(f"Turn {turn_number} | hist_store={hist_count} turns")
        print(f"Q: {question}")
        print(f"A: {answer[:500]}")
        print(f"   [{latency:.0f}ms | {len(doc_hits)} doc + {len(hist_hits)} hist chunks]")
        print()

    return {
        'question'  : question,
        'answer'    : answer,
        'doc_hits'  : len(doc_hits),
        'hist_hits' : len(hist_hits),
        'latency_ms': latency,
    }


print("multi_turn_rag() defined.")


## Step 9 — Multi-Turn Demo

Six questions across two topic threads. Later turns benefit from semantic history
retrieval even when the relevant prior turn is several exchanges back.


In [ ]:
SESSION = "conv_session_1"

questions = [
    # Thread A — NWP
    "What is Numerical Weather Prediction (NWP)?",
    # Thread B — observation network
    "What types of observation instruments are used to collect weather data?",
    # Thread A resumed — refers back to NWP implicitly
    "What are the main sources of error in those atmospheric models?",
    # Thread B resumed — refers back to observation data
    "How is the data from satellites and buoys transmitted to forecasting centres?",
    # Cross-thread — connects both
    "How does the quality of observation data affect the accuracy of NWP forecasts?",
    # Deep callback — tests history retrieval from turn 1
    "What equations govern the NWP model described earlier?",
]

results = []
print("=" * 70)
for i, q in enumerate(questions, 1):
    r = multi_turn_rag(q, SESSION, turn_number=i, verbose=True)
    results.append(r)
    print("-" * 70)


## Step 9b — Retrieval Analysis

In [ ]:
hist_total = qdrant.get_collection(HIST_COLLECTION).points_count or 0
doc_total  = qdrant.get_collection(DOC_COLLECTION).points_count or 0

print(f"Session          : {SESSION}")
print(f"Q&A turns        : {len(results)}")
print(f"Doc store size   : {doc_total} vectors")
print(f"Hist store size  : {hist_total} vectors")
print(f"Avg latency      : {sum(r['latency_ms'] for r in results)/len(results):.0f}ms")
print()
print(f"{'Turn':<6} {'Doc hits':>9} {'Hist hits':>10} {'Latency':>9}  Question")
print("-" * 75)
for i, r in enumerate(results, 1):
    print(f"{i:<6} {r['doc_hits']:>9} {r['hist_hits']:>10} {r['latency_ms']:>8.0f}ms  {r['question'][:45]}")


## Step 10 — Summary

### What Multi-Turn Conversational RAG does differently

| Dimension | NB 23 Memory-Augmented | NB 24 Multi-Turn Conversational |
|-----------|----------------------|--------------------------------|
| History storage | Python list (RAM) | Qdrant collection (persistent) |
| History retrieval | Sliding window (last N) | Semantic search over full history |
| History capacity | Bounded by window size | Unlimited |
| Relevance of old turns | Degrades after window | Retrieved if semantically relevant |
| Two-store architecture | No | Yes — doc + hist searched in parallel |

### Dual-store architecture

```
query_vector
  ├─► conv_rag_24_docs    → top-4 document chunks  (factual ground truth)
  └─► conv_rag_24_history → top-2 history turns    (session continuity)
        ↓
      merge → LLM prompt → answer → embed & store in history
```

### pdfplumber word-level extraction

| Step | Detail |
|------|--------|
| `page.extract_words()` | Returns list of `{text, x0, top, ...}` word dicts |
| Line grouping | Sort by `round(top / y_tol)` to bin words into rows |
| Column handling | x0 sort within each row keeps multi-column order |
| Chunking | Lines joined to paragraph, then `recursive_split()` |

> **Next: [25 — Ensemble RAG](../tier6_ensemble_meta/25_Ensemble_RAG.ipynb)** —
> run Simple + HyDE + Fusion in parallel, aggregate with RRF, return the best-supported answer.


In [ ]:
print(f"Doc collection  '{DOC_COLLECTION}' : {doc_total} vectors")
print(f"Hist collection '{HIST_COLLECTION}': {hist_total} vectors")
print(f"PDF framework   : pdfplumber extract_words() — word-level line grouping")
print(f"RAG pattern     : Multi-Turn Conversational RAG — dual Qdrant stores, semantic history")
print()
print("Notebook 24 — Multi-Turn Conversational RAG complete.")
